## Example 1: hatchling finds the plugin's version source, build hook and metadata hook

In [ ]:
import importlib.metadata
from hatch_vcs import hooks
from hatch_vcs.build_hook import VCSBuildHook
from hatch_vcs.metadata_hook import VCSMetadataHook
from hatch_vcs.version_source import VCSVersionSource

# hatchling loads the module named by the "hatch" entry point and calls its hook functions.
(entry_point,) = [ep for ep in importlib.metadata.entry_points(group="hatch") if ep.value == "hatch_vcs.hooks"]
print(f"entry point: {entry_point.name} = {entry_point.value}")
assert entry_point.load() is hooks

assert hooks.hatch_register_version_source() is VCSVersionSource
assert hooks.hatch_register_build_hook() is VCSBuildHook
assert hooks.hatch_register_metadata_hook() is VCSMetadataHook
for plugin in (VCSVersionSource, VCSBuildHook, VCSMetadataHook):
    print(f"{plugin.__name__}: PLUGIN_NAME = {plugin.PLUGIN_NAME!r}")
    assert plugin.PLUGIN_NAME == "vcs"

## Example 2: outside a repository, the fallback version is used

In [ ]:
import tempfile
from hatch_vcs.version_source import VCSVersionSource

# A directory that is not a git checkout has no tags to read.
with tempfile.TemporaryDirectory() as root:
    source = VCSVersionSource(root, {"fallback-version": "1.2.3"})
    data = source.get_version_data()
    print(f"version data: {data}")
    assert data == {"version": "1.2.3"}

    # Without a fallback version there is nothing to report, and the lookup fails.
    try:
        VCSVersionSource(root, {}).get_version_data()
    except LookupError as error:
        print(f"no fallback: {type(error).__name__}")
    else:
        raise AssertionError("a version was reported for a directory without version control")

## Example 3: the plugin options become setuptools-scm settings

In [ ]:
import tempfile
from hatch_vcs.version_source import VCSVersionSource

with tempfile.TemporaryDirectory() as root:
    source = VCSVersionSource(root, {
        "tag-pattern": r"^release-(?P<version>[0-9.]+)$",
        "fallback-version": "0.0.1",
        "raw-options": {"local_scheme": "no-local-version", "write_to": "ignored.py"},
    })
    config = source.construct_setuptools_scm_config()
    print(f"setuptools-scm config: {config}")
    assert config == {
        "local_scheme": "no-local-version",
        "root": root,
        "tag_regex": r"^release-(?P<version>[0-9.]+)$",
        "fallback_version": "0.0.1",
    }  # write_to is dropped: the build hook is the one that writes files

    # Options of the wrong type are rejected.
    for bad in ({"tag-pattern": 5}, {"fallback-version": 1}, {"raw-options": ["local_scheme"]}):
        try:
            VCSVersionSource(root, bad).construct_setuptools_scm_config()
        except TypeError as error:
            print(f"rejected {bad}: {error}")
        else:
            raise AssertionError(f"accepted {bad}")

## Example 4: the version comes from the latest git tag

In [ ]:
import os
import subprocess
import tempfile
from hatch_vcs.version_source import VCSVersionSource

def git(root, *args):
    # A committer identity and signing switched off, so this works without any git configuration.
    command = ["git", "-c", "user.name=Example", "-c", "user.email=example@example.com",
               "-c", "commit.gpgsign=false", "-c", "tag.gpgsign=false", *args]
    return subprocess.run(command, cwd=root, check=True, capture_output=True, text=True).stdout

with tempfile.TemporaryDirectory() as root:
    git(root, "init", "-q")
    with open(os.path.join(root, "README.md"), "w") as f:
        f.write("first\n")
    git(root, "add", "README.md")
    git(root, "commit", "-q", "-m", "first")
    git(root, "tag", "v0.4.0")

    tagged = VCSVersionSource(root, {}).get_version_data()["version"]
    print(f"on the tag: {tagged}")
    assert tagged == "0.4.0"

    # One commit after the tag: a development release of the next patch version.
    with open(os.path.join(root, "README.md"), "a") as f:
        f.write("second\n")
    git(root, "commit", "-q", "-am", "second")
    head = git(root, "rev-parse", "--short", "HEAD").strip()
    after = VCSVersionSource(root, {}).get_version_data()["version"]
    plain = VCSVersionSource(root, {"raw-options": {"local_scheme": "no-local-version"}}).get_version_data()["version"]
    print(f"one commit later: {after}  (without the local part: {plain})")
    assert plain == "0.4.1.dev1"
    assert after.startswith("0.4.1.dev1+g") and after[len("0.4.1.dev1+g"):].startswith(head[:7]), (after, head)

## Example 5: a custom tag pattern, and project URLs that carry the commit hash

In [ ]:
import os
import subprocess
import tempfile
from hatch_vcs.metadata_hook import VCSMetadataHook
from hatch_vcs.version_source import VCSVersionSource

def git(root, *args):
    # A committer identity and signing switched off, so this works without any git configuration.
    command = ["git", "-c", "user.name=Example", "-c", "user.email=example@example.com",
               "-c", "commit.gpgsign=false", "-c", "tag.gpgsign=false", *args]
    return subprocess.run(command, cwd=root, check=True, capture_output=True, text=True).stdout

with tempfile.TemporaryDirectory() as root:
    git(root, "init", "-q")
    with open(os.path.join(root, "README.md"), "w") as f:
        f.write("release\n")
    git(root, "add", "README.md")
    git(root, "commit", "-q", "-m", "release")
    git(root, "tag", "release-2.0.0")

    # tag-pattern extracts the version from tags that do not look like versions.
    source = VCSVersionSource(root, {"tag-pattern": r"^release-(?P<version>[0-9.]+)$"})
    version = source.get_version_data()["version"]
    print(f"version from tag release-2.0.0: {version}")
    assert version == "2.0.0"

    # The metadata hook fills {commit_hash} in the project URLs.
    head = git(root, "rev-parse", "HEAD").strip()
    metadata = {}
    hook = VCSMetadataHook(root, {"urls": {"Source": "https://example.com/project/tree/{commit_hash}"}})
    hook.update(metadata)
    print(f"urls: {metadata['urls']}")
    assert metadata["urls"] == {"Source": f"https://example.com/project/tree/{head}"}